# InferStack N2: where does native prefill time go?

Runtime > Change runtime type > **L4 GPU** (same GPU as the 2026-11-13 scorecard, so the numbers line up) or A100. Run the cells in order. About 30-45 minutes on L4; no vLLM/SGLang install.

**Diagnosis only.** No source or profile is changed and nothing here is a scored speed number. The scorecard showed native prefill is the largest gap (prefill_heavy TTFT p50 at 16 users: native 3.6-4.1 s, SGLang 0.16 s). This notebook loads the native engine in-process, feeds it the scorecard's own prefill_heavy prompts (~1500 words) and splits prefill time into:

- **model**: forward passes,
- **snapshot**: prefix-cache copies taken every 128 prompt tokens (single-request cohorts only),
- **other**: sampling, accept and Python.

It repeats that across cohort size (1, 4), chunk size (128 = current setting, 512, 2048) and prefix cache on/off, for native_unfused and native_full, then takes a torch.profiler trace of one prefill. Comparing with the served TTFT shows how much is scheduler waiting.

At the end download the ZIP and bring it back.

## 1. Verify GPU, clone privately, pin source

In [ ]:
import base64, datetime, getpass, json, os, signal, subprocess, sys, time
from pathlib import Path
PLATFORM = 'colab'
COMMIT = '476bf3fabb3b18fb31d0ae6cc4e04629ce58e3cf'
BASE = Path('/content' if PLATFORM == 'colab' else '/kaggle/working')
REPO = BASE / 'inferstack'
assert (3, 11) <= sys.version_info[:2] < (3, 14), 'This profile needs Python 3.11–3.13.'
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True)
print(gpu)
if PLATFORM == 'colab':
    # One GPU, BF16-capable (compute capability >= 8.0). A100 preferred; L4 accepted.
    # The label goes into every output path so an L4 run can never be filed as A100.
    GPU_LABEL = 'a100' if 'A100' in gpu else 'l4' if 'L4' in gpu else None
    assert GPU_LABEL, 'Choose an A100 or L4 runtime. T4 has no BF16; use the separate T4 notebook for Kaggle.'
else:
    assert len([line for line in gpu.splitlines() if 'T4' in line]) >= 2, 'Choose Kaggle GPU T4 x2.'
    GPU_LABEL = 't4x2'
print('GPU label:', GPU_LABEL)

def clone_pinned():
    if not (REPO / '.git').is_dir():
        if PLATFORM == 'kaggle':
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('GITHUB_TOKEN').strip()
        else:
            token = getpass.getpass('Read-only GitHub token for this private repo: ').strip()
        env = os.environ.copy()
        env.update(GIT_TERMINAL_PROMPT='0', GIT_CONFIG_COUNT='1',
                   GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                   GIT_CONFIG_VALUE_0='AUTHORIZATION: basic '+base64.b64encode(('x-access-token:'+token).encode()).decode())
        result = subprocess.run(['git','clone','-q','https://github.com/sanjoxtech/inferstack.git',str(REPO)],
                                env=env,capture_output=True,text=True)
        del token, env
        if result.returncode:
            raise RuntimeError('Clone failed. Check read-only repository access and Internet connectivity. Credentials were not printed or stored.')
    subprocess.run(['git','checkout','--detach',COMMIT],cwd=REPO,check=True,capture_output=True,text=True)
    actual=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
    assert actual == COMMIT
    print('Pinned source:', actual)
clone_pinned()
RUN_ID=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=REPO/'results'/f'{PLATFORM}-{GPU_LABEL}-n2-prefill-{COMMIT[:7]}-{RUN_ID}'
OUT.mkdir(parents=True,exist_ok=False)
(OUT/'gpu-inventory.txt').write_text(gpu)
(OUT/'run-context.json').write_text(json.dumps({'commit':COMMIT,'platform':PLATFORM,'gpu_label':GPU_LABEL,'run_id':RUN_ID},indent=2))

def logged(argv, filename, *, env=None, timeout=3600):
    path=OUT/filename
    start=time.monotonic()
    with path.open('w') as log:
        proc=subprocess.Popen(list(map(str,argv)),cwd=REPO,env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        try:
            while proc.poll() is None:
                elapsed=time.monotonic()-start
                if elapsed > timeout:
                    raise TimeoutError(f'Wall-time cap reached: {filename}')
                print(f'{filename}: running, {int(elapsed)} seconds elapsed',flush=True)
                time.sleep(20)
        except BaseException:
            try:
                os.killpg(proc.pid,signal.SIGTERM)
                proc.wait(timeout=20)
            except (ProcessLookupError,subprocess.TimeoutExpired):
                try: os.killpg(proc.pid,signal.SIGKILL)
                except ProcessLookupError: pass
            raise
    print(filename,'exit:',proc.returncode)
    if proc.returncode:
        print(path.read_text(errors='replace')[-3000:])
    return proc.returncode


## 2. Native InferStack environment

In [ ]:
# Reuse the notebook's CUDA-enabled torch; keep changes in a separate virtualenv.
# Colab/Kaggle images may lack the venv package for their current Python (e.g. 3.13):
# install the matching pythonX.Y-venv if creating a virtualenv fails.
def ensure_venv():
    probe=subprocess.run([sys.executable,'-m','venv','--help'],capture_output=True)
    ok=subprocess.run([sys.executable,'-c','import ensurepip'],capture_output=True).returncode==0
    if probe.returncode==0 and ok: return
    assert os.geteuid()==0, 'venv/ensurepip missing and no root to install it'
    pkg=f'python{sys.version_info.major}.{sys.version_info.minor}-venv'
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq',pkg],check=True)
ensure_venv()
VENV=Path('/tmp/inferstack-native-env')
subprocess.run([sys.executable,'-m','venv','--system-site-packages',str(VENV)],check=True)
PY=VENV/'bin/python'
ENV={**os.environ,'PATH':str(VENV/'bin')+os.pathsep+os.environ['PATH'],
     'PYTHONPATH':str(REPO/'src'),'INFERSTACK_PROVIDER':'colab','CUDA_VISIBLE_DEVICES':'0'}
assert logged([PY,'-m','pip','install','--require-hashes','-r','requirements-dev.lock'],
              'install-core.log',env=ENV,timeout=2700)==0
assert logged([PY,'-m','pip','install','--no-cache-dir','-e','.[dev,native,hf,postgres]',
               'transformers[kernels]==5.22.2'], 'install-native.log',env=ENV,timeout=2700)==0
preflight="""import json, torch, triton, transformers
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,'transformers':transformers.__version__,
                  'triton':triton.__version__,'gpu':torch.cuda.get_device_name(0)},indent=2))
"""
assert logged([PY,'-c',preflight],'preflight-native.log',env=ENV,timeout=180)==0
print((OUT/'preflight-native.log').read_text())
(OUT/'pip-freeze-native.txt').write_text(subprocess.check_output([str(PY),'-m','pip','freeze'],env=ENV,text=True))


## 3. Prefill phase split (both native profiles)

In [ ]:
# N2 runs: each one starts a fresh process, loads the model, measures, exits.
PROFILES_N2 = {'native_unfused': 'configs/profiles/production-v1.6.0-baseline.toml',
               'native_full': 'configs/profiles/production-v1.6.0.toml'}
def gpu_used_mib():
    return int(subprocess.check_output(['nvidia-smi','--query-gpu=memory.used','--format=csv,noheader,nounits'],text=True).split()[0])
def n2(name, *args):
    assert gpu_used_mib() <= 1024, f'GPU busy ({gpu_used_mib()} MiB); Runtime > Restart session and re-run from cell 1'
    code = logged([PY,'-u','scripts/n2_prefill_diagnosis.py',*args,'--out',str((OUT/f'{name}.json').relative_to(REPO))],
                  f'{name}.log', env=ENV, timeout=3*60*60)
    print(name, 'OK' if code == 0 else f'FAILED (exit {code}); continuing')
    return code
RESULTS = {}
for prof, path in PROFILES_N2.items():
    RESULTS[f'phases-{prof}'] = n2(f'phases-{prof}', 'phases', '--profile', path,
                                   '--batches', '1', '4', '--chunks', '128', '512', '2048', '--repeats', '3')
print(RESULTS)

## 4. Operator traces

In [ ]:
# Operator trace of one prefill at the current settings (chunk 128, prefix cache on).
for prof, path in PROFILES_N2.items():
    for b in (1, 4):
        name = f'ops-{prof}-b{b}'
        RESULTS[name] = n2(name, 'ops', '--profile', path, '--batch', str(b), '--chunk', '128',
                           '--trace', str((OUT/f'{name}-trace.json').relative_to(REPO)))
print(RESULTS)

## 5. Summary

In [ ]:
# Summary: prefill time split, and served TTFT from the L4 scorecard for comparison.
SERVED_TTFT_L4 = {('native_unfused',1): 1135.6, ('native_unfused',4): 1009.8,
                  ('native_full',1): 1016.6, ('native_full',4): 898.5}   # prefill_heavy TTFT p50 ms, concurrency 1 / 4
lines = ['| profile | batch | chunk | prefix cache | wall ms | model ms | snapshot ms | other ms | forwards | served TTFT p50 (L4) |',
         '|---|---|---|---|---|---|---|---|---|---|']
for prof in PROFILES_N2:
    p = OUT/f'phases-{prof}.json'
    if not p.exists(): lines.append(f'| {prof} | missing: see phases-{prof}.log |'); continue
    for r in json.loads(p.read_text())['rows']:
        if 'error' in r:
            lines.append(f"| {prof} | {r['batch']} | {r['chunk']} | {r['prefix_cache']} | error: {r['error'][:60]} |"); continue
        served = SERVED_TTFT_L4.get((prof, r['batch'])) if GPU_LABEL == 'l4' and r['chunk'] == 128 and r['prefix_cache'] else None
        lines.append(f"| {prof} | {r['batch']} | {r['chunk']} | {r['prefix_cache']} | {r['wall_ms']:.0f} [{r['min_wall_ms']:.0f}-{r['max_wall_ms']:.0f}] | "
                     f"{r['model_ms']:.0f} | {r['snapshot_ms']:.0f} | {r['other_ms']:.0f} | {r['model_calls']} | {'' if served is None else served} |")
table = '\n'.join(lines)
(OUT/'n2-summary.md').write_text(table + '\n')
print(table)
for prof in PROFILES_N2:
    p = OUT/f'ops-{prof}-b1.json'
    if p.exists():
        d = json.loads(p.read_text())
        print(f'\nTop GPU ops, {prof}, batch 1:')
        for o in d['top_device_ops'][:12]: print(f"  {o['self_device_pct']:5.1f}%  {o['self_device_us']/1000:8.1f} ms  x{o['count']:<5} {o['name'][:90]}")

## 6. Download evidence

In [ ]:
import shutil
archive_base=BASE/f'inferstack-{PLATFORM}-{GPU_LABEL}-n2-{RUN_ID}'
zip_path=shutil.make_archive(str(archive_base),'zip',root_dir=OUT.parent,base_dir=OUT.name)
print('Evidence archive:',zip_path)
if PLATFORM=='colab':
    from google.colab import files
    files.download(zip_path)
else:
    from IPython.display import FileLink, display
    display(FileLink(zip_path))
    print('You can also download the ZIP from the notebook file browser under /kaggle/working.')
